In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/silver_utils

In [0]:
# Cell 1: Setup, Context & Environment Initialization
db_helpers = DatabricksHelpers(dbutils)
current_user_email = db_helpers.current_user()

catalog_name = "f1_dev"
schemas = ["bronze", "silver", "gold"]

# Ensure bronze, silver, and gold schemas exist in Unity Catalog
initialize_environment(spark, catalog_name, schemas)

source_table = f"{catalog_name}.bronze.results"
target_table = f"{catalog_name}.silver.fact_results"

# --- CELL 1 TEST ---
assert current_user_email and ("@" in current_user_email or current_user_email == "root"), \
    f"TEST FAILED: Invalid user context derived: '{current_user_email}'"

assert spark.catalog.tableExists(source_table), \
    f"TEST FAILED: Source bronze table '{source_table}' does not exist in catalog."

print(f"✓ Cell 1 Passed: Executed by '{current_user_email}' on catalog '{catalog_name}'.")

In [0]:
# Cell 2: Data Extraction & Bronze String Sanitization
df_bronze = spark.table(source_table)

# Reusable string cleaning: purges '\N', 'NULL', 'null', and empty strings across all text fields
df_sanitized = sanitize_bronze_strings(df_bronze)

# --- CELL 2 TEST ---
string_cols = [f.name for f in df_sanitized.schema.fields if f.dataType.typeName() == "string"]
for col_name in string_cols:
    dirty_count = df_sanitized.filter(
        (F.col(col_name) == r"\N") | (F.lower(F.col(col_name)) == "null")
    ).count()
    assert dirty_count == 0, f"TEST FAILED: Found {dirty_count} unhandled string literals in '{col_name}'"

print(f"✓ Cell 2 Passed: String sanitization verified across {len(string_cols)} string columns.")

In [0]:
# Cell 3: Silver Domain Transformations & Safe Casting
df_results_silver = (
    df_sanitized
    .select(
        F.col("resultId").try_cast("integer").alias("result_id"),
        F.col("raceId").try_cast("integer").alias("race_id"),
        F.col("driverId").try_cast("integer").alias("driver_id"),
        F.col("constructorId").try_cast("integer").alias("constructor_id"),
        F.col("number").try_cast("integer").alias("car_number"),
        F.col("grid").try_cast("integer").alias("grid_position"),
        F.col("position").try_cast("integer").alias("position"),
        F.col("positionText").alias("position_text"),
        F.col("positionOrder").try_cast("integer").alias("position_order"),
        F.col("points").try_cast("double").alias("points"),
        F.col("laps").try_cast("integer").alias("laps_completed"),
        F.col("time").alias("race_time"),
        F.col("milliseconds").try_cast("long").alias("race_time_ms"),
        F.col("fastestLap").try_cast("integer").alias("fastest_lap_number"),
        F.col("rank").try_cast("integer").alias("fastest_lap_rank"),
        F.col("fastestLapTime").alias("fastest_lap_time"),
        F.col("fastestLapSpeed").try_cast("double").alias("fastest_lap_speed_kph"),
        F.col("statusId").try_cast("integer").alias("status_id")
    )
    # Deduplicate against primary key
    .dropDuplicates(["result_id"])
)

# --- CELL 3 TEST ---
sample_row = df_results_silver.first()

# 1. Primary & Foreign Key data type verification
assert dict(df_results_silver.dtypes)["result_id"] == "int", "TEST FAILED: result_id cast failed."
assert dict(df_results_silver.dtypes)["race_id"] == "int", "TEST FAILED: race_id cast failed."
assert dict(df_results_silver.dtypes)["driver_id"] == "int", "TEST FAILED: driver_id cast failed."
assert dict(df_results_silver.dtypes)["constructor_id"] == "int", "TEST FAILED: constructor_id cast failed."

# 2. Metric data type verification
assert dict(df_results_silver.dtypes)["points"] == "double", "TEST FAILED: points cast failed."
assert dict(df_results_silver.dtypes)["race_time_ms"] == "bigint", "TEST FAILED: race_time_ms cast failed."

print("✓ Cell 3 Passed: Fact results transformations and safe casts verified.")

In [0]:
# Cell 4: Pre-Write Data Quality Circuit Breaker & Domain Rules
# 1. Standard Circuit Breaker: PK Non-Null & Uniqueness check
run_circuit_breaker(
    df=df_results_silver, 
    primary_keys=["result_id"], 
    prohibited_pii=set()
)

# 2. Fact Domain Integrity Checks
# Foreign Keys must not be NULL
assert df_results_silver.filter(F.col("race_id").isNull()).count() == 0, \
    "CIRCUIT BREAKER: Found NULL race_id foreign keys!"
assert df_results_silver.filter(F.col("driver_id").isNull()).count() == 0, \
    "CIRCUIT BREAKER: Found NULL driver_id foreign keys!"
assert df_results_silver.filter(F.col("constructor_id").isNull()).count() == 0, \
    "CIRCUIT BREAKER: Found NULL constructor_id foreign keys!"

# Non-negative metrics rule
negative_points = df_results_silver.filter(F.col("points") < 0).count()
assert negative_points == 0, f"CIRCUIT BREAKER: Found {negative_points} records with negative points!"

negative_laps = df_results_silver.filter(F.col("laps_completed") < 0).count()
assert negative_laps == 0, f"CIRCUIT BREAKER: Found {negative_laps} records with negative laps!"

print("✓ Cell 4 Passed: Pre-write circuit breaker and F1 fact domain assertions passed.")

In [0]:
# Cell 5: Target Delta DDL Creation & Idempotent Delta Load
# 1. Ensure target Silver Fact Delta table exists with Primary & Foreign Key constraints
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        result_id INT NOT NULL,
        race_id INT NOT NULL,
        driver_id INT NOT NULL,
        constructor_id INT NOT NULL,
        car_number INT,
        grid_position INT,
        position INT,
        position_text STRING,
        position_order INT,
        points DOUBLE,
        laps_completed INT,
        race_time STRING,
        race_time_ms BIGINT,
        fastest_lap_number INT,
        fastest_lap_rank INT,
        fastest_lap_time STRING,
        fastest_lap_speed_kph DOUBLE,
        status_id INT,
        CONSTRAINT fact_results_pk PRIMARY KEY (result_id)
    )
    USING DELTA
""")

# 2. Reusable Upsert: Enables CDF, Auto-Optimize, and executes SCD Type 1 MERGE
upsert_silver_delta(
    spark=spark, 
    df=df_results_silver, 
    target_table=target_table, 
    primary_keys=["result_id"]
)

# --- CELL 5 TEST ---
target_df = spark.table(target_table)
staging_count = df_results_silver.count()
target_count = target_df.count()

# Confirm written count matches staging expectations
assert target_count >= staging_count, \
    f"TEST FAILED: Target table count ({target_count}) is lower than staging count ({staging_count})."

print(f"✓ Cell 5 Passed: Delta MERGE load to '{target_table}' completed and verified.")

In [0]:
# Cell 6: Unity Catalog Metadata Governance & Exit
# Reusable Governance Tagger: Sets tags and asserts existence in information_schema
apply_silver_tags(
    spark=spark, 
    active_catalog=catalog_name, 
    target_table=target_table, 
    current_user_email=current_user_email, 
    table_type="fact"
)

# Clean exit for Databricks Workflows / Jobs Orchestration
dbutils.notebook.exit(f"SUCCESS: Pipeline executed cleanly by {current_user_email} on {target_table}")